In [2]:
import numpy as np
import cv2
from ultralytics import YOLO


class yolov8_detector:
    def __init__(self, model_path=r"d:\project\step1\week12\yolov8n.pt"):
        self.module = YOLO(model_path)

    def get_result(self, frame):
        """
        输入一帧 BGR 图像，返回 [{bbox, conf, cls_}, ...]；无目标返回 None
        """
        out_results = []
        result = self.module.predict(frame, conf=0.4, iou=0.45, verbose=False)[0]

        bboxes = result.boxes.xyxy.cpu().numpy()   # (N, 4)  x1, y1, x2, y2
        confs = result.boxes.conf.cpu().numpy()    # (N,)    置信度
        cls_idx = result.boxes.cls.cpu().numpy()   # (N,)    类别索引

        for bbox, conf, cls_ in zip(bboxes, confs, cls_idx):
            x1, y1, x2, y2 = map(int, bbox)
            out_results.append({"bbox": [x1, y1, x2, y2],
                                "conf": float(conf),
                                "cls_": int(cls_)})

        return out_results if len(out_results) > 0 else None


if __name__ == "__main__":
    # 读 car.mp4 视频逐帧检测
    detector = yolov8_detector()
    cap = cv2.VideoCapture(r"d:\project\step1\week13\car.mp4")

    frame_idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        results = detector.get_result(frame)
        # 每隔 60 帧打印一次（约每 2 秒），避免 3411 帧全部刷屏
        if frame_idx % 60 == 0:
            n = len(results) if results else 0
            ids = ", ".join(f"cls{r['cls_']}(conf {r['conf']:.2f})" for r in results or [])
            print(f"帧 {frame_idx:>4}: {n} 个目标 -> {ids or '无目标'}")
        frame_idx += 1
    cap.release()

    print(f"\n共处理 {frame_idx} 帧")


帧    0: 3 个目标 -> cls0(conf 0.89), cls0(conf 0.66), cls2(conf 0.46)
帧   60: 3 个目标 -> cls0(conf 0.83), cls0(conf 0.77), cls2(conf 0.47)
帧  120: 2 个目标 -> cls2(conf 0.81), cls1(conf 0.47)
帧  180: 2 个目标 -> cls2(conf 0.81), cls2(conf 0.45)
帧  240: 3 个目标 -> cls2(conf 0.90), cls2(conf 0.84), cls2(conf 0.49)
帧  300: 3 个目标 -> cls2(conf 0.90), cls2(conf 0.61), cls2(conf 0.58)
帧  360: 2 个目标 -> cls0(conf 0.78), cls0(conf 0.77)
帧  420: 2 个目标 -> cls2(conf 0.70), cls2(conf 0.47)

共处理 430 帧


In [3]:
import numpy as np
import cv2


class IOU_tracker:
    """简单 IoU 多目标跟踪器：新检测与已有轨迹按 IoU 贪心匹配"""
    # COCO 类别索引 -> 名称（只保留需要计数的类别）
    COCO_NAMES = {
        0: "person", 1: "bicycle", 2: "car", 3: "motorcycle",
        5: "bus", 7: "truck",
    }

    def __init__(self, iou_thresh=0.3, max_lost=5):
        self.yolov8_detector = yolov8_detector()
        self.tracks = []                # 活动轨迹 [{tracker_id, bbox, conf, cls_, lost}]
        self.next_id = 1                # 下一个新 ID
        self.iou_thresh = iou_thresh    # 匹配 IoU 阈值
        self.max_lost = max_lost        # 连续多少帧未匹配则删除轨迹
        self.cls_count = {name: 0 for name in self.COCO_NAMES.values()}  # 各类别累计计数

    @staticmethod
    def compute_iou(box1, box2):
        """计算两个 [x1, y1, x2, y2] 框的交并比 IoU"""
        x1 = max(box1[0], box2[0]); y1 = max(box1[1], box2[1])
        x2 = min(box1[2], box2[2]); y2 = min(box1[3], box2[3])
        inter = max(0, x2 - x1) * max(0, y2 - y1)          # 交集面积
        area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
        area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
        union = area1 + area2 - inter                       # 并集面积
        return inter / union if union > 0 else 0.0

    def generate_tracker(self, frame_data):
        """处理一帧：检测 -> IoU 匹配 -> 更新/新建/删除轨迹，返回当前帧轨迹列表"""
        new_det = self.yolov8_detector.get_result(frame_data) or []   # 无目标时为 []

        keep = []
        matched = set()

        # 1) 贪心匹配：每个新检测找一个 IoU 最大且超过阈值的老轨迹
        for det in new_det:
            best_iou, best_idx = self.iou_thresh, -1
            for j, track in enumerate(self.tracks):
                if j in matched:                            # 已被占用的轨迹跳过
                    continue
                iou = self.compute_iou(track["bbox"], det["bbox"])
                if iou > best_iou:
                    best_iou, best_idx = iou, j

            if best_idx >= 0:                               # 匹配成功：沿用老 ID，更新框
                t = self.tracks[best_idx]
                t.update(bbox=det["bbox"], conf=det["conf"], cls_=det["cls_"], lost=0)
                keep.append(t)
                matched.add(best_idx)
            else:                                           # 新目标：分配新 ID，类别计数 +1
                cls_name = self.COCO_NAMES.get(det["cls_"])
                if cls_name is not None:
                    self.cls_count[cls_name] += 1
                keep.append({"tracker_id": self.next_id, "bbox": det["bbox"],
                             "conf": det["conf"], "cls_": det["cls_"], "lost": 0})
                self.next_id += 1

        # 2) 未匹配的老轨迹：lost 计数，超过 max_lost 帧才删除（容忍短暂遮挡/漏检）
        for j, track in enumerate(self.tracks):
            if j not in matched:
                track["lost"] += 1
                if track["lost"] <= self.max_lost:
                    keep.append(track)

        self.tracks = keep
        return keep

    def _vis(self, frame):
        """在帧上绘制轨迹框、ID 和各类别实时计数"""
        for t in self.tracks:
            x1, y1, x2, y2 = t["bbox"]
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(frame, f"ID {t['tracker_id']}", (x1, max(y1 - 5, 15)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
        # 左上角显示各类别累计计数
        y = 22
        for name, n in self.cls_count.items():
            cv2.putText(frame, f"{name}: {n}", (10, y),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)
            y += 22

    def run(self, video_path):
        """读取视频逐帧跟踪并实时显示（按 q 退出）"""
        cap = cv2.VideoCapture(video_path)
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            self.generate_tracker(frame)
            self._vis(frame)

            cv2.imshow("IOU Tracker", frame)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
        cap.release()
        cv2.destroyAllWindows()
        print("各类别目标计数:", self.cls_count)


# ---- 演示：读取 car.mp4 视频逐帧跟踪 ----
if __name__ == "__main__":
    tracker = IOU_tracker()
    cap = cv2.VideoCapture(r"d:\project\step1\week13\car.mp4")

    frame_idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        tracks = tracker.generate_tracker(frame)
        # 每隔 60 帧打印一次（约每 2 秒），避免 3411 帧全部刷屏
        if frame_idx % 60 == 0:
            ids = ", ".join(f"ID{t['tracker_id']}(conf {t['conf']:.2f})" for t in tracks)
            print(f"帧 {frame_idx:>4}: {len(tracks)} 个目标 -> {ids or '无目标'}")
        frame_idx += 1
    cap.release()

    print(f"\n共处理 {frame_idx} 帧，共分配 {tracker.next_id - 1} 个 ID")
    print("各类别目标计数:", tracker.cls_count)


帧    0: 3 个目标 -> ID1(conf 0.89), ID2(conf 0.66), ID3(conf 0.46)
帧   60: 6 个目标 -> ID1(conf 0.83), ID2(conf 0.77), ID17(conf 0.47), ID15(conf 0.52), ID16(conf 0.44), ID14(conf 0.43)
帧  120: 3 个目标 -> ID27(conf 0.81), ID28(conf 0.47), ID2(conf 0.76)
帧  180: 2 个目标 -> ID30(conf 0.81), ID32(conf 0.45)
帧  240: 3 个目标 -> ID34(conf 0.90), ID30(conf 0.84), ID37(conf 0.49)
帧  300: 4 个目标 -> ID34(conf 0.90), ID30(conf 0.61), ID43(conf 0.58), ID44(conf 0.43)
帧  360: 2 个目标 -> ID59(conf 0.78), ID58(conf 0.77)
帧  420: 2 个目标 -> ID64(conf 0.70), ID67(conf 0.47)

共处理 430 帧，共分配 69 个 ID
各类别目标计数: {'person': 19, 'bicycle': 5, 'car': 35, 'motorcycle': 3, 'bus': 0, 'truck': 2}


# 🎯 单元格 2 代码流程讲解：`IOU_tracker` 多目标跟踪

## 整体流程图

```mermaid
flowchart TD
    A["① 初始化 IOU_tracker()<br/>yolov8_detector + tracks=[]<br/>next_id=1 + 各类别计数=0"] --> B["② run(video_path)<br/>逐帧读取视频"]
    B --> C["③ generate_tracker(frame)"]
    C --> D["④ yolov8_detector.get_result(frame)<br/>得到 new_det 检测列表"]
    D --> E["⑤ 对每个新检测 det：<br/>在『未匹配』的旧轨迹中找 IoU 最大者"]
    E --> F{"⑥ 最大 IoU > iou_thresh ?"}
    F -- "是 ✅ 匹配成功" --> G["⑦ 沿用旧 tracker_id<br/>更新 bbox / conf / cls_<br/>lost 清零"]
    F -- "否 🆕 新目标" --> H["⑧ 分配新 ID (next_id+1)<br/>类别计数 cls_count += 1"]
    G --> I["⑨ 加入 keep"]
    H --> I
    I --> J["⑩ 处理未匹配的旧轨迹：lost + 1"]
    J --> K{"⑪ lost > max_lost ?"}
    K -- "是" --> L["⑫ 删除该轨迹"]
    K -- "否" --> M["⑬ 继续保留（容忍漏检/遮挡）"]
    L --> N["⑭ self.tracks = keep"]
    M --> N
    N --> O["⑮ 返回当前帧轨迹列表"]
    O --> P["⑯ _vis(frame)：<br/>画跟踪框 + ID + 左上角实时计数"]
    P --> B
```

## 关键机制说明

| 机制 | 说明 |
|------|------|
| **IoU 匹配** | 新检测与旧轨迹框的交并比，超过阈值（默认 0.3）视为同一目标 |
| **ID 沿用** | 匹配成功则保留旧 ID，保证目标身份跨帧稳定 |
| **新 ID 分配** | 无法匹配的检测视为新目标，`next_id += 1` |
| **类别计数** | 只在分配新 ID 时 `cls_count[类别] += 1`，避免重复计数 |
| **丢失容忍** | 未匹配轨迹 `lost += 1`，超过 `max_lost`（默认 5）才删除 |
| **可视化** | 每帧绘制跟踪框、ID 号、左上角各类别累计数 |

## 一句话总结

> 每帧循环：**检测 → IoU 贪心匹配 → 更新 / 新建 / 删除轨迹 → 可视化**。
> 计数只发生在"新建轨迹"时，所以统计的是**出现的不同目标个数**，而不是每帧目标数 —— 这正是单元格 4 计数暴涨的修复思路。


In [4]:
import cv2

tracker = IOU_tracker()
cap = cv2.VideoCapture(r"d:\project\step1\week13\car.mp4")
fps = cap.get(cv2.CAP_PROP_FPS)
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# 输出带跟踪标注的视频
out_path = r"d:\project\step1\week13\track_out.mp4"
writer = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

frame_idx = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break
    tracks = tracker.generate_tracker(frame)
    # 画框 + ID
    for t in tracks:
        x1, y1, x2, y2 = t["bbox"]
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(frame, f"ID {t['tracker_id']}", (x1, max(y1 - 5, 15)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
    writer.write(frame)
    frame_idx += 1
cap.release()
writer.release()
print(f"已生成带跟踪标注的视频: {out_path}（{frame_idx} 帧, {w}x{h}, {fps:.0f}fps）")

# 在 Notebook 内联播放
from IPython.display import Video
Video(out_path, width=480)


已生成带跟踪标注的视频: d:\project\step1\week13\track_out.mp4（430 帧, 720x1280, 30fps）
